# Form Coach — Notebook 01: Pose Inference**Deliverable 1: Core Vision Tasks & Inference (25 pts)**This notebook loads an Ultralytics YOLO **pose estimation** model and runs real inferenceon my own recorded footage. Pose is a task beyond plain detection, which is what therubric asks for.What it produces:- Keypoint tensors from a real model on a real video- A joint-angle time series computed from those keypoints- An annotated output video> Run every cell top to bottom, then download this file **with the output still in it**.

## 1. Environment Setup

In [ ]:
%pip install -q ultralyticsimport ultralyticsultralytics.checks()

In [ ]:
import cv2import osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom ultralytics import YOLOfrom IPython.display import Video, Image as IPImageprint("Imports OK")

## 2. Configuration**Set these three values to match your footage, then everything else adapts.**`SIDE` matters: the camera only sees one side of your body clearly. The far-sidekeypoints are occluded and their confidence collapses, which makes the angles noisy.Set it to the side facing the camera.

In [ ]:
# ---- EDIT THESE ----------------------------------------------------VIDEO_PATH = "pushups.mp4"   # your filmed clipSIDE       = "left"          # "left" or "right" — the side facing the cameraEXERCISE   = "pushup"        # "pushup" or "squat"TARGET_FPS = 5               # process this many frames per second# --------------------------------------------------------------------POSE_MODEL = "yolo26n-pose.pt"# COCO-17 keypoint indices used by YOLO pose modelsKEYPOINTS = {    "nose": 0,    "left_shoulder": 5,  "right_shoulder": 6,    "left_elbow": 7,     "right_elbow": 8,    "left_wrist": 9,     "right_wrist": 10,    "left_hip": 11,      "right_hip": 12,    "left_knee": 13,     "right_knee": 14,    "left_ankle": 15,    "right_ankle": 16,}# Which three joints form the angle we track, per exerciseCHAIN = {    "pushup": ["shoulder", "elbow", "wrist"],    "squat":  ["hip", "knee", "ankle"],}joint_names = [f"{SIDE}_{j}" for j in CHAIN[EXERCISE]]KPT_IDS = [KEYPOINTS[n] for n in joint_names]print(f"Exercise : {EXERCISE}")print(f"Side     : {SIDE}")print(f"Tracking : {joint_names}")print(f"kpts     : {KPT_IDS}   <-- these are the indices AIGym needs in notebook 02")

## 3. Upload your videoRun this cell and pick your clip from your computer. Skip it if you've alreadymounted Drive or the file is present.

In [ ]:
if not os.path.exists(VIDEO_PATH):    from google.colab import files    uploaded = files.upload()    VIDEO_PATH = list(uploaded.keys())[0]    print(f"Uploaded: {VIDEO_PATH}")else:    print(f"Found existing file: {VIDEO_PATH}")cap = cv2.VideoCapture(VIDEO_PATH)assert cap.isOpened(), f"Could not open {VIDEO_PATH}"W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))FPS = cap.get(cv2.CAP_PROP_FPS)N  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))cap.release()print(f"Resolution : {W} x {H}")print(f"FPS        : {FPS}")print(f"Frames     : {N}  (~{N/FPS:.1f} s)")

## 4. Load the pose model and run inference on one frameStarting with a single frame proves the model works and lets us inspect the rawkeypoint tensor before processing the whole clip.

In [ ]:
model = YOLO(POSE_MODEL)cap = cv2.VideoCapture(VIDEO_PATH)ok, first_frame = cap.read()cap.release()assert ok, "Could not read first frame"results = model.predict(source=first_frame, verbose=False)result = results[0]print(f"People detected: {len(result.boxes)}")if result.keypoints is not None and len(result.keypoints) > 0:    print(f"Keypoints tensor shape (persons, keypoints, xy): {result.keypoints.xy.shape}")    print(f"Confidence tensor shape: {result.keypoints.conf.shape}")else:    print("No keypoints found — check framing: the whole body must be visible.")annotated = result.plot()cv2.imwrite("frame_pose.jpg", annotated)IPImage(filename="frame_pose.jpg", width=700)

### Confidence checkBefore processing the whole video, verify the side you chose is actually the visible one.If the near-side confidences aren't clearly higher, flip `SIDE` and re-run.

In [ ]:
if result.keypoints is not None and len(result.keypoints) > 0:    conf = result.keypoints.conf[0].cpu().numpy()    rows = []    for joint in CHAIN[EXERCISE]:        rows.append({            "joint": joint,            "left_conf":  round(float(conf[KEYPOINTS[f"left_{joint}"]]), 3),            "right_conf": round(float(conf[KEYPOINTS[f"right_{joint}"]]), 3),        })    df_conf = pd.DataFrame(rows)    print(df_conf.to_string(index=False))    left_mean  = df_conf["left_conf"].mean()    right_mean = df_conf["right_conf"].mean()    better = "left" if left_mean > right_mean else "right"    print(f"\nHigher-confidence side: {better}  (you set SIDE = '{SIDE}')")    if better != SIDE:        print("--> Consider flipping SIDE and re-running from the config cell.")

## 5. Joint angle from keypoints`AIGym` counts reps internally, but it doesn't hand you the angle. Computing itourselves gives a measurable form signal: how deep each rep actually went.The angle at joint **b**, between segments **b→a** and **b→c**, comes from the dot product.

In [ ]:
def joint_angle(a, b, c):    """Angle in degrees at point b, formed by points a-b-c."""    a, b, c = np.asarray(a, float), np.asarray(b, float), np.asarray(c, float)    ba, bc = a - b, c - b    denom = np.linalg.norm(ba) * np.linalg.norm(bc)    if denom < 1e-6:        return np.nan    cosine = np.clip(np.dot(ba, bc) / denom, -1.0, 1.0)    return float(np.degrees(np.arccos(cosine)))# sanity check: a right angleprint("90 deg test:", round(joint_angle([0, 1], [0, 0], [1, 0]), 1))

## 6. Process the full videoFrame sampling keeps this fast — we process `TARGET_FPS` frames per second instead ofall of them. Same pattern as Lab 2.

In [ ]:
MIN_CONF = 0.5   # ignore keypoints the model isn't sure abouttemp_out  = "pose_raw.mp4"final_out = "01_pose_annotated.mp4"cap = cv2.VideoCapture(VIDEO_PATH)skip_rate = max(1, int(round(FPS / TARGET_FPS)))writer = cv2.VideoWriter(temp_out, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (W, H))records = []frame_idx = 0processed = 0while cap.isOpened():    ok, frame = cap.read()    if not ok:        break    if frame_idx % skip_rate == 0:        res = model.predict(source=frame, verbose=False)[0]        annotated = res.plot()        angle_val = np.nan        if res.keypoints is not None and len(res.keypoints) > 0:            xy   = res.keypoints.xy[0].cpu().numpy()            conf = res.keypoints.conf[0].cpu().numpy()            if all(conf[i] >= MIN_CONF for i in KPT_IDS):                angle_val = joint_angle(*[xy[i] for i in KPT_IDS])        if not np.isnan(angle_val):            cv2.putText(annotated, f"{EXERCISE} angle: {angle_val:.0f} deg",                        (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)        records.append({            "frame": frame_idx,            "time_s": round(frame_idx / FPS, 2),            "angle": angle_val,        })        writer.write(annotated)        processed += 1    frame_idx += 1cap.release()writer.release()df = pd.DataFrame(records)valid = df["angle"].notna().sum()print(f"Frames read      : {frame_idx}")print(f"Frames processed : {processed}")print(f"Angle captured   : {valid}/{len(df)} ({100*valid/max(len(df),1):.0f}%)")if valid:    print(f"Angle range      : {df['angle'].min():.0f} to {df['angle'].max():.0f} deg")

In [ ]:
# Re-encode to H.264 so the notebook can play itif os.path.exists(temp_out):    os.system(f"ffmpeg -i {temp_out} -vcodec libx264 -f mp4 {final_out} -y -loglevel quiet")Video(final_out, embed=True, width=800)

## 7. The angle traceEach dip in this curve is one rep. Reps that don't reach the same depth as the othersare the ones with incomplete range of motion — this is the form signal.**Note the min and max here.** Notebook 02 needs them to set the AIGym rep thresholds.

In [ ]:
if df["angle"].notna().sum() > 0:    plt.figure(figsize=(13, 4))    plt.plot(df["time_s"], df["angle"], linewidth=1.8)    plt.xlabel("Time (s)")    plt.ylabel("Joint angle (degrees)")    plt.title(f"{EXERCISE.capitalize()} — {joint_names[1]} angle over time")    plt.grid(alpha=0.3)    plt.tight_layout()    plt.savefig("01_angle_trace.png", dpi=120)    plt.show()    print(f"Suggested AIGym thresholds for notebook 02:")    print(f"  down_angle = {df['angle'].min() + 10:.0f}")    print(f"  up_angle   = {df['angle'].max() - 10:.0f}")else:    print("No valid angles. Try lowering MIN_CONF, or re-film with the full body in frame.")df.to_csv("01_angle_data.csv", index=False)print("\nSaved: 01_angle_data.csv")

## 8. Deliverable 1 evidenceProduced by this notebook:| Artifact | File ||---|---|| Annotated pose video | `01_pose_annotated.mp4` || Single-frame proof | `frame_pose.jpg` || Angle trace plot | `01_angle_trace.png` || Raw angle data | `01_angle_data.csv` |**Rubric mapping:** a YOLO model loaded and real inference run via the Python API on myown video, with results captured. Pose estimation is the task beyond plain detection,using task-specific weights (`yolo26n-pose.pt`).### Before you close this tab1. File → Download → Download .ipynb (**after** running, output intact)2. Commit the notebook, the PNG and the CSV3. Write down the `down_angle` / `up_angle` values above — notebook 02 needs them